# 🌿 Git: guardando a história do seu trabalho

Segunda-feira, 9h. Você abre a pasta de rede e encontra:

```
relatorio_inadimplencia.xlsx
relatorio_inadimplencia_v2.xlsx
relatorio_inadimplencia_v2_final.xlsx
relatorio_inadimplencia_v2_final_AGORA_VAI.xlsx
relatorio_inadimplencia_v2_final_AGORA_VAI_revisado_Maria.xlsx
```

O gestor pergunta: *"Qual desses é o que eu aprovei na sexta? E o que mudou da versão anterior?"*

Ninguém sabe responder com segurança. O **Git** existe para resolver exatamente isso: ele guarda **cada versão do trabalho, quem mexeu, quando e por quê**, sem precisar de cinco cópias do mesmo arquivo.

Hoje não vou ensinar vocês a decorar comandos. Vou mostrar **como o Git pensa**, fazer uma demonstração ao vivo e deixar o caminho aberto para quem quiser usar no dia a dia.

## 🗺️ Roteiro

- 🧠 As ideias-chave: repositório, commit, histórico, branch, merge e repositório remoto
- 🐙 Criar a conta no GitHub
- 💻 Demonstração local: `init`, `status`, `add`, `commit`, `log`
- ☁️ Demonstração remota: `push` e `pull`
- 🔎 Repositórios famosos: navegar no `pandas` e fazer um *fork*
- ↩️ Desfazer e conflitos (só para ver como é)

**No fim da aula você vai saber:**
1. explicar o que é um repositório e um commit, com suas palavras;
2. ler o histórico de um projeto no GitHub;
3. reconhecer o fluxo `pull → add → commit → push`;
4. ter uma conta no GitHub e um *fork* de um projeto real;
5. saber quando vale a pena usar Git e quando não vale.

## ⚙️ Configuração

O Git é um programa do computador, não uma biblioteca do Python. Vou chamá-lo de dentro do notebook com `!`, que manda o comando para o terminal. Primeiro, vamos conferir se ele está instalado.

In [1]:
# Esta célula mostra a versão do Git instalada no computador.
!git --version

git version 2.56.0.windows.2


Se apareceu um número de versão, está tudo certo. 💡 O `!` na frente significa "isto não é Python, é um comando do terminal".

## 🔁 Revisão: onde o jeito antigo trava

O jeito que todo mundo já conhece é **copiar o arquivo e mudar o nome**. Vou simular isso: criar uma pasta com as cópias do relatório e olhar o resultado.

In [2]:
# Esta célula cria uma pasta de exemplo com as cópias de um relatório, do jeito antigo.
from pathlib import Path

pasta_antiga = Path("exemplo_versoes")
pasta_antiga.mkdir(parents=True, exist_ok=True)

nomes = [
    "relatorio_inadimplencia.xlsx",
    "relatorio_inadimplencia_v2.xlsx",
    "relatorio_inadimplencia_v2_final.xlsx",
    "relatorio_inadimplencia_v2_final_AGORA_VAI.xlsx",
]
for nome in nomes:
    (pasta_antiga / nome).write_text("conteúdo de exemplo")

sorted(arquivo.name for arquivo in pasta_antiga.iterdir())

['relatorio_inadimplencia.xlsx',
 'relatorio_inadimplencia_v2.xlsx',
 'relatorio_inadimplencia_v2_final.xlsx',
 'relatorio_inadimplencia_v2_final_AGORA_VAI.xlsx']

São **4 arquivos** para uma única planilha. Olhando só os nomes, não dá para saber:

- qual é a versão oficial;
- **o que** mudou de uma para a outra;
- **quem** mudou e **por quê**;
- como voltar para a versão de sexta-feira se alguém estragou a de hoje.

E isso com uma pessoa. Com três pessoas editando ao mesmo tempo, vira bagunça.

## 🧠 As ideias-chave

São poucas palavras, e todas têm uma analogia do escritório.

| Palavra | O que é | Analogia |
|---|---|---|
| **Repositório** | A pasta do projeto, com todo o histórico dentro | A pasta de rede do projeto, mas com memória |
| **Commit** | Uma "foto" do projeto num momento, com uma mensagem explicando o que mudou | Salvar uma versão aprovada, escrevendo na capa "o que mudou" |
| **Histórico** | A lista de todos os commits, do mais novo ao mais antigo | O livro-razão do projeto |
| **Branch** (ramo) | Uma linha paralela de trabalho, para testar sem mexer na versão oficial | Fazer uma cópia para rascunho, mas com controle |
| **Merge** (mesclar) | Juntar o trabalho de um ramo de volta à versão oficial | Incorporar as mudanças do rascunho ao relatório final |
| **Repositório remoto** | A cópia do repositório num servidor (GitHub), que todos acessam | A pasta de rede compartilhada |

**A ideia central:** você não sobrescreve nada. Cada commit é uma foto nova, e todas as fotos antigas continuam guardadas. Por isso dá para voltar no tempo.

### Os três lugares por onde o arquivo passa

Antes de virar commit, o arquivo passa por três lugares. Guarde esta imagem, ela explica quase todos os comandos:

```
  sua pasta de trabalho   ──  git add  ──▶   área de preparação   ──  git commit  ──▶   histórico
  (onde você edita)                          (o que vai na foto)                        (as fotos guardadas)
```

Pense numa foto de família: primeiro você **escolhe quem entra** no enquadramento (`add`) e só depois **bate a foto** (`commit`). Isso permite fotografar só parte do que mudou.

## 🐙 Criando a conta no GitHub

Até agora falei do Git, que é o programa que guarda o histórico **no seu computador**. O **GitHub** é o site onde as pessoas guardam uma cópia desse histórico na internet, para compartilhar e trabalhar em conjunto. Vamos criar a conta agora, juntos.

**Passo a passo:**
1. Abra **github.com** e clique em **Sign up** (Cadastre-se).
2. Informe um **e-mail**, uma **senha** e um **nome de usuário**.
3. Resolva a verificação de que você não é um robô.
4. O GitHub envia um **código por e-mail**. Digite o código para confirmar.
5. Escolha o plano **Free** (gratuito). É o que vamos usar.

💡 **Três cuidados, pensando em quem trabalha em banco:**
- Use o seu **e-mail pessoal**, não o corporativo. A conta é sua e continua com você se mudar de empresa.
- Escolha um nome de usuário que você não se importe de mostrar num currículo, porque ele aparece publicamente.
- **Nunca suba dados de clientes, bases ou código da empresa para um repositório público.** O que vai para a internet pode ser copiado e nunca mais some. Hoje vamos usar só dados de exemplo.

Quem já tem conta pode seguir para a próxima seção.

## 💻 Demonstração local: o ciclo do Git

Agora vou mostrar o Git funcionando. Vamos acompanhar o **dia a dia de um analista**: criar uma pasta de projeto, guardar a primeira versão da planilha de inadimplência e depois uma segunda versão, com uma mudança.

Vou usar uma pasta de treino, criada ao lado do notebook. Primeiro, preparo uma pasta limpa.

In [3]:
# Esta célula apaga a pasta de treino, se ela já existir, e cria uma pasta vazia.
import os
import shutil
import stat
from pathlib import Path

pasta_treino = Path("projeto_treino")

def liberar_e_apagar(funcao, caminho, _):
    # O Git deixa alguns arquivos somente leitura; liberamos para poder apagar.
    os.chmod(caminho, stat.S_IWRITE)
    funcao(caminho)

if pasta_treino.exists():
    shutil.rmtree(pasta_treino, onerror=liberar_e_apagar)
pasta_treino.mkdir(parents=True)
os.chdir(pasta_treino)
print("Estou em:", Path.cwd().name)

Estou em: projeto_treino


### Passo 1: `git init` (criar o repositório)

Esse comando transforma a pasta num **repositório**. Daqui para frente, o Git passa a observar tudo o que acontece nela.

In [4]:
# Esta célula transforma a pasta de treino em um repositório Git.
!git init -b main

Initialized empty Git repository in H:/Ada- Tech/Tecnicas de Programacao/aula-03-pandas-fundamentos/projeto_treino/.git/


O Git criou uma pasta escondida chamada `.git`. É lá dentro que ele guarda todo o histórico. **Você nunca mexe nela à mão.**

Falta uma configuração: o Git precisa saber **quem é você**, porque cada commit leva o nome do autor. Normalmente isso é feito uma vez só por computador (com `--global`). Aqui vou configurar só para este repositório de treino.

In [5]:
# Esta célula informa ao Git o nome e o e-mail do autor, só para este repositório de treino.
!git config user.name "Analista Exemplo"
!git config user.email "analista@exemplo.com"
!git config user.name

Analista Exemplo


### Passo 2: `git status` (o que está acontecendo?)

Esse é o comando que você mais vai usar. Ele responde: *"o que mudou desde a última foto?"*

In [6]:
# Esta célula pergunta ao Git o estado do repositório, que ainda está vazio.
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


O repositório está vazio: ainda não há nenhum commit e nenhum arquivo. Vou criar a primeira planilha, uma tabela pequena de inadimplência por agência.

In [7]:
# Esta célula cria o arquivo inadimplencia.csv com três agências.
conteudo = """agencia,clientes,inadimplentes
Centro,1200,60
Zona Norte,800,56
Zona Sul,950,38
"""
Path("inadimplencia.csv").write_text(conteudo, encoding="utf-8")
print(conteudo)

agencia,clientes,inadimplentes
Centro,1200,60
Zona Norte,800,56
Zona Sul,950,38



In [8]:
# Esta célula pergunta de novo o estado: agora existe um arquivo novo.
!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	inadimplencia.csv

nothing added to commit but untracked files present (use "git add" to track)


Repare na palavra **untracked** (não rastreado): o arquivo existe na pasta, mas o Git ainda **não está acompanhando** esse arquivo. Ele só vai guardar o que a gente mandar.

### Passo 3: `git add` (escolher quem entra na foto)

Lembra da foto de família? O `add` é o momento de **escolher quem entra no enquadramento**.

In [9]:
# Esta célula coloca o arquivo na área de preparação.
!git add inadimplencia.csv
!git status

On branch main



No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   inadimplencia.csv



Agora aparece **Changes to be committed** (mudanças prontas para virar commit): o arquivo está na área de preparação, no enquadramento.

### Passo 4: `git commit` (bater a foto)

O commit guarda a foto no histórico. A parte `-m` é a **mensagem**, onde você escreve o que fez.

💡 **Uma boa mensagem diz o que mudou, para alguém que não estava na sala.**

| ❌ Evite | ✅ Prefira |
|---|---|
| `alteracoes` | `Adiciona inadimplência por agência` |
| `v2 final` | `Corrige total de clientes da Zona Sul` |
| `ajustes` | `Inclui coluna de taxa de inadimplência` |

In [10]:
# Esta célula guarda a primeira versão no histórico, com uma mensagem.
!git commit -m "Adiciona inadimplência por agência"


[main (root-commit) 40f2039] Adiciona inadimplência por agência
 1 file changed, 4 insertions(+)
 create mode 100644 inadimplencia.csv


In [11]:
# Esta célula confere o estado depois do commit.
!git status

On branch main
nothing to commit, working tree clean


Dessa vez a mensagem é `nothing to commit, working tree clean`: **nada a commitar, pasta de trabalho limpa**. A foto foi guardada e nada mudou desde então. 💡 O Git responde em inglês; as palavras que mais aparecem são `untracked`, `staged` / `to be committed` e `clean`.

### Passo 5: `git log` (o histórico)

Esse comando lista as fotos guardadas. Com `--oneline` ele mostra uma linha por commit.

In [12]:
# Esta célula lista o histórico de commits, uma linha por commit.
!git log --oneline

40f2039 Adiciona inadimplência por agência


Há **1 commit**. O código ao lado (os primeiros caracteres) é o **identificador** da foto: não existem dois iguais.

Agora vou fazer uma mudança: a gerência avisou que a Zona Norte tem mais 2 inadimplentes do que o informado.

In [13]:
# Esta célula corrige o número de inadimplentes da Zona Norte no arquivo.
conteudo = conteudo.replace("Zona Norte,800,56", "Zona Norte,800,58")
Path("inadimplencia.csv").write_text(conteudo, encoding="utf-8")
print(conteudo)

agencia,clientes,inadimplentes
Centro,1200,60
Zona Norte,800,58
Zona Sul,950,38



### Passo 6: `git diff` (o que mudou, linha a linha)

Aqui está o ganho em relação às cópias `_v2_final`: o Git mostra **exatamente o que mudou**. Linhas com `-` foram removidas e com `+` foram adicionadas.

In [14]:
# Esta célula mostra a diferença entre o arquivo atual e o último commit.
!git diff

diff --git a/inadimplencia.csv b/inadimplencia.csv
index ddd4cae..96eeb7c 100644
--- a/inadimplencia.csv
+++ b/inadimplencia.csv
@@ -1,4 +1,4 @@
 agencia,clientes,inadimplentes
 Centro,1200,60
-Zona Norte,800,56
+Zona Norte,800,58
 Zona Sul,950,38


Mudou uma linha só: `56` virou `58`. Ninguém precisa abrir duas planilhas lado a lado. Vamos guardar essa segunda versão.

In [15]:
# Esta célula prepara o arquivo e guarda a segunda versão, com uma mensagem.
!git add inadimplencia.csv
!git commit -m "Corrige inadimplentes da Zona Norte"


[main 1ecc309] Corrige inadimplentes da Zona Norte
 1 file changed, 1 insertion(+), 1 deletion(-)


In [16]:
# Esta célula lista o histórico completo, com autor e data de cada commit.
!git log

commit 1ecc309c607e5279621263c92b0f25e9d5042964
Author: Analista Exemplo <analista@exemplo.com>
Date:   Mon Nov 9 08:46:47 2026 -0300

    Corrige inadimplentes da Zona Norte

commit 40f20392164811575395f8b95bb960692a6530c0
Author: Analista Exemplo <analista@exemplo.com>
Date:   Mon Nov 9 08:46:47 2026 -0300

    Adiciona inadimplência por agência


Agora o histórico mostra **2 commits**, e cada um traz o **autor**, a **data** e a **mensagem**. Essa é a resposta à pergunta do gestor: *quem mudou, quando e por quê.*

✅ **O ciclo local, resumido:** `status` (olhar) → `add` (escolher) → `commit` (guardar) → `log` (conferir).

## ↩️ Desfazer: o Git como rede de segurança

Aqui está uma das maiores vantagens do Git. Imagine que alguém bagunçou a planilha sem querer. Vou simular: apagar duas agências do arquivo.

In [17]:
# Esta célula estraga o arquivo de propósito, deixando só uma agência.
Path("inadimplencia.csv").write_text("agencia,clientes,inadimplentes\nCentro,1200,60\n", encoding="utf-8")
!git status

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   inadimplencia.csv

no changes added to commit (use "git add" and/or "git commit -a")


O Git avisa `modified`: o arquivo mudou desde o último commit. Para **descartar a mudança e voltar ao último commit**, uso o `git restore`.

In [18]:
# Esta célula desfaz a mudança e devolve o arquivo ao estado do último commit.
!git restore inadimplencia.csv
print(Path("inadimplencia.csv").read_text(encoding="utf-8"))

agencia,clientes,inadimplentes
Centro,1200,60
Zona Norte,800,58
Zona Sul,950,38


As três agências voltaram, com o `58` da Zona Norte que corrigimos. Nada se perdeu, porque o commit guardava a foto.

💡 **Dois comandos de desfazer, para não confundir:**
- `git restore arquivo`: descarta as mudanças **ainda não commitadas** (como acabamos de fazer). Cuidado: o que não foi commitado não volta.
- `git restore --staged arquivo`: tira o arquivo da área de preparação, **sem perder a edição**. Foi um `git add` por engano? É este.

Em apostilas mais antigas você vai ver `git checkout -- arquivo` e `git reset HEAD arquivo`. São os nomes antigos desses dois comandos, e ainda funcionam.

### Um erro comum

**Vou provocar o erro de propósito:** tentar adicionar um arquivo que não existe, como acontece quando se erra o nome.

In [19]:
# Esta célula tenta adicionar um arquivo que não existe, para ver a mensagem de erro.
!git add planilha_que_nao_existe.csv

fatal: pathspec 'planilha_que_nao_existe.csv' did not match any files


**Como ler:** `fatal: pathspec 'planilha_que_nao_existe.csv' did not match any files`. Traduzindo: "o caminho que você me deu não corresponde a nenhum arquivo". Quase sempre é nome digitado errado, ou você está na pasta errada. O `git status` ajuda a conferir.

## ☁️ O repositório remoto: `push` e `pull`

Até aqui tudo ficou no seu computador. O repositório **remoto** é a cópia no GitHub, que permite duas coisas:

- **Backup:** se o computador quebrar, o trabalho está lá.
- **Colaboração:** outras pessoas baixam, alteram e devolvem.

Dois comandos fazem o transporte:

| Comando | O que faz | Analogia |
|---|---|---|
| `git push` | **Envia** seus commits para o GitHub | Colocar a pasta na rede compartilhada |
| `git pull` | **Traz** para o seu computador o que mudou no GitHub | Baixar a versão que o colega atualizou |

💡 **Regra de ouro do dia a dia:** `pull` **antes** de começar a trabalhar e `push` **depois** de commitar. É o fluxo `pull → add → commit → push`.

Esta parte eu faço no terminal do meu computador, para vocês acompanharem pelo GitHub. Os comandos que vou usar são estes:

```bash
# 1. Conectar a pasta local ao repositório vazio criado no GitHub
git remote add origin https://github.com/SEU_USUARIO/projeto-treino.git

# 2. Enviar os commits (o -u lembra o caminho para as próximas vezes)
git push -u origin main

# 3. Depois de alguém alterar algo no GitHub, trazer a novidade
git pull
```

**Roteiro da demonstração:** crio um repositório vazio no site do GitHub, conecto a pasta com `remote add`, envio com `push`, edito um arquivo **direto no site** e trago a mudança de volta com `pull`.

⚠️ O GitHub **não aceita a senha da conta** no terminal. O login é feito pelo navegador (o Git abre uma janela na primeira vez) ou com um *token*. Por isso o `push` é demonstração: o importante hoje é entender o caminho que o trabalho faz.

## 🔎 Repositórios de verdade

Um repositório é só um projeto com histórico. Vamos olhar projetos reais.

### Navegando no `pandas`

Abra no navegador: **github.com/pandas-dev/pandas**. A biblioteca que usaremos na segunda parte da aula é um repositório aberto. Procurem:

- **Commits:** o histórico completo, com milhares de mensagens, cada uma dizendo o que mudou;
- **Branches:** as linhas paralelas em que as pessoas trabalham;
- **Contributors:** as pessoas de empresas e países diferentes que mantêm o projeto;
- **Issues** e **Pull requests:** onde se pedem e se propõem mudanças.

Isso funciona porque cada pessoa trabalha no seu ramo e as mudanças são mescladas depois (*merge*) à versão principal.

### Fork e clone com o `Spoon-Knife`

O GitHub mantém um repositório pequeno só para treinar, o **octocat/Spoon-Knife**. Dois conceitos novos:

| Palavra | O que faz |
|---|---|
| **Fork** | Faz uma **cópia do repositório de outra pessoa para a sua conta** no GitHub, pelo site |
| **Clone** | **Baixa** um repositório do GitHub para o computador, com todo o histórico |

**Para fazer agora, pelo site:** abra **github.com/octocat/Spoon-Knife** e clique em **Fork**. O repositório aparece na sua conta.

Já o *clone* podemos fazer aqui no notebook, porque o repositório é público. Primeiro, saio da pasta de treino.

In [20]:
# Esta célula sai da pasta de treino e volta para a pasta do notebook.
os.chdir("..")
print("Estou em:", Path.cwd().name)

Estou em: aula-03-pandas-fundamentos


In [21]:
# Esta célula baixa (clona) o repositório Spoon-Knife para uma pasta ao lado do notebook.
pasta_clone = Path("Spoon-Knife")
if pasta_clone.exists():
    shutil.rmtree(pasta_clone, onerror=liberar_e_apagar)

!git clone https://github.com/octocat/Spoon-Knife.git

Cloning into 'Spoon-Knife'...


In [22]:
# Esta célula lista os arquivos que vieram no clone.
sorted(p.name for p in pasta_clone.iterdir() if p.name != ".git")

['README.md', 'index.html', 'styles.css']

In [23]:
# Esta célula mostra os três commits mais recentes do projeto clonado.
!git -C Spoon-Knife log --oneline -n 3

d0dd1f6 Pointing to the guide for forking
bb4cc8d Create styles.css and updated README
a30c19e Created index page for future collaborative edits


Com um comando, trouxemos o **projeto inteiro com o histórico**. Qualquer pessoa pode fazer isso com um repositório público, e é assim que as equipes começam a trabalhar em cima do mesmo código.

## ⚔️ Conflito: quando duas pessoas mexem na mesma linha

Se a Ana e o Bruno editam **a mesma linha** do mesmo arquivo e depois tentam juntar, o Git não sabe qual vale e pede ajuda. Isso se chama **conflito**. O arquivo fica assim:

```
agencia,clientes,inadimplentes
<<<<<<< HEAD
Zona Norte,800,58
=======
Zona Norte,800,60
>>>>>>> ramo-do-bruno
```

**Como ler:** entre `<<<<<<<` e `=======` está a **sua** versão, e entre `=======` e `>>>>>>>` a **do outro**. Você abre o arquivo, **escolhe o valor certo**, apaga as linhas com `<<<`, `===` e `>>>`, e faz o commit.

💡 Conflito não é erro grave, é o Git pedindo uma decisão que só uma pessoa sabe tomar. O jeito de diminuir os conflitos é o `pull` antes de começar e commits pequenos e frequentes.

## 🌿 Branch e merge, em duas linhas

Uma **branch** é uma linha paralela de trabalho. A principal chama-se `main`. Para testar uma ideia sem mexer na versão oficial: crie uma branch, trabalhe nela e, se der certo, faça o **merge** de volta à `main`.

```bash
git switch -c teste-nova-coluna     # cria a branch e vai para ela (antes: git checkout -b)
git switch main                     # volta para a principal (antes: git checkout main)
git merge teste-nova-coluna         # traz o trabalho da branch para a main
```

Deixei um passo a passo para praticar no notebook complementar.

## 🧠 Questões para pensar

### ❓ Questão 1

Qual é a diferença entre **Git** e **GitHub**?

*Pense primeiro. A resposta está logo abaixo, escondida:*

<details>
<summary>👀 Ver a resposta</summary>

O **Git** é o programa que guarda o histórico no seu computador. O **GitHub** é um site que guarda uma cópia desse histórico na internet para compartilhar. Dá para usar o Git sem o GitHub, mas não o contrário.

</details>

### ❓ Questão 2

Você editou duas planilhas, mas só quer guardar uma no próximo commit. O que você faz?

<details>
<summary>👀 Ver a resposta</summary>

Faz o `git add` **só da planilha que quer guardar** e depois o `git commit`. A outra continua modificada, mas fora da foto. É para isso que existe a área de preparação.

</details>

### ❓ Questão 3

O que o `git status` mostra e por que ele é o comando mais usado?

<details>
<summary>👀 Ver a resposta</summary>

Mostra o que mudou desde o último commit e o que já está preparado para o próximo. Ele responde "em que pé estou?" antes de qualquer outra ação, e evita erros.

</details>

### ❓ Questão 4

Você apagou uma agência da planilha sem querer, mas ainda não fez commit. Como volta ao que estava?

<details>
<summary>👀 Ver a resposta</summary>

Com `git restore nome_do_arquivo`, que devolve o arquivo ao estado do último commit.

</details>

### ❓ Questão 5

Por que mensagens como `ajustes` e `v2 final` são ruins num commit?

<details>
<summary>👀 Ver a resposta</summary>

Porque o histórico existe para responder **o que mudou e por quê**. Com mensagens vagas, ele vira de novo a pasta de `_v2_final_AGORA_VAI`. Uma boa mensagem faz sentido para quem não estava na sala.

</details>

### ❓ Questão 6

Dois colegas mudaram a mesma linha e deu conflito. O Git errou?

<details>
<summary>👀 Ver a resposta</summary>

Não. O Git não sabe qual valor está certo, então pede para uma pessoa decidir. Você escolhe o valor, remove as marcas `<<<<<<<`, `=======` e `>>>>>>>` e faz o commit.

</details>

### ❓ Questão 7

Na empresa, você pode subir a base de clientes para um repositório público do GitHub, para o colega acessar de casa?

<details>
<summary>👀 Ver a resposta</summary>

**Não.** Repositório público é visível por qualquer pessoa, e o que é publicado pode ser copiado e nunca mais some. Dados de clientes e da empresa seguem as regras de segurança do banco e não vão para contas pessoais.

</details>

## 🧾 Resumo

### Comandos

| Comando | Para que serve |
|---|---|
| `git init` | Transforma a pasta num repositório |
| `git status` | Mostra o que mudou e o que está preparado |
| `git add arquivo` | Coloca o arquivo na área de preparação |
| `git commit -m "mensagem"` | Guarda a foto no histórico |
| `git log --oneline` | Lista os commits, um por linha |
| `git diff` | Mostra o que mudou, linha a linha |
| `git restore arquivo` | Descarta mudanças não commitadas (antes: `git checkout -- arquivo`) |
| `git restore --staged arquivo` | Tira o arquivo da área de preparação (antes: `git reset HEAD arquivo`) |
| `git remote add origin URL` | Liga a pasta a um repositório no GitHub |
| `git push` | Envia os commits para o GitHub |
| `git pull` | Traz as novidades do GitHub |
| `git clone URL` | Baixa um repositório inteiro |
| `git switch -c nome` | Cria uma branch e vai para ela (antes: `git checkout -b nome`) |
| `git switch nome` | Muda de branch (antes: `git checkout nome`) |
| `git merge nome` | Traz o trabalho de uma branch para a atual |

### Mensagens que aparecem, e como ler

| Mensagem | O que significa |
|---|---|
| `Untracked files` | Arquivo novo que o Git ainda não acompanha |
| `Changes to be committed` | Arquivos preparados com `add`, prontos para o commit |
| `modified` | Arquivo conhecido que mudou desde o último commit |
| `nothing to commit, working tree clean` | Nada mudou desde o último commit |
| `fatal: pathspec ... did not match any files` | O nome do arquivo não existe, ou você está na pasta errada |
| `fatal: not a git repository` | Você não está dentro de um repositório (falta o `git init`, ou é a pasta errada) |
| `CONFLICT (content)` | Duas pessoas mudaram a mesma linha, é preciso escolher o valor |

## 🏋️ Exercícios para casa

Cada exercício usa a pasta `exercicio_casa`. Escreva sua resposta na célula vazia e compare com a resposta recolhida.

### Exercício 1: criar um repositório e o primeiro commit

Crie a pasta `exercicio_casa`, transforme-a em repositório, configure nome e e-mail, crie o arquivo `metas.csv` com as linhas `vendedor,meta` e `Ana,50000`, e faça o primeiro commit com uma boa mensagem.

*Dica: `Path("exercicio_casa").mkdir(exist_ok=True)` cria a pasta, e `git -C exercicio_casa ...` roda o comando dentro dela.*

In [24]:
# sua resposta aqui


In [25]:
# Resposta
pasta_exercicio = Path("exercicio_casa")
if pasta_exercicio.exists():
    shutil.rmtree(pasta_exercicio, onerror=liberar_e_apagar)
pasta_exercicio.mkdir()

!git -C exercicio_casa init -b main
!git -C exercicio_casa config user.name "Analista Exemplo"
!git -C exercicio_casa config user.email "analista@exemplo.com"
(pasta_exercicio / "metas.csv").write_text("vendedor,meta\nAna,50000\n", encoding="utf-8")
!git -C exercicio_casa add metas.csv
!git -C exercicio_casa commit -m "Adiciona metas de vendas"


Initialized empty Git repository in H:/Ada- Tech/Tecnicas de Programacao/aula-03-pandas-fundamentos/exercicio_casa/.git/


[main (root-commit) cb1f0e3] Adiciona metas de vendas
 1 file changed, 2 insertions(+)
 create mode 100644 metas.csv


### Exercício 2: ver o que mudou

Altere a meta da Ana para `55000` no arquivo, veja o que mudou com `git diff`, e guarde a mudança num segundo commit.

*Dica: reescreva o arquivo com `write_text`, como fizemos na aula.*

In [26]:
# sua resposta aqui


In [27]:
# Resposta
(pasta_exercicio / "metas.csv").write_text("vendedor,meta\nAna,55000\n", encoding="utf-8")
!git -C exercicio_casa diff
!git -C exercicio_casa add metas.csv
!git -C exercicio_casa commit -m "Atualiza meta da Ana para 55000"


diff --git a/metas.csv b/metas.csv
index dac3978..f00aeb2 100644
--- a/metas.csv
+++ b/metas.csv
@@ -1,2 +1,2 @@
 vendedor,meta
-Ana,50000
+Ana,55000


[main e72264c] Atualiza meta da Ana para 55000
 1 file changed, 1 insertion(+), 1 deletion(-)


### Exercício 3: desfazer e conferir o histórico

Estrague o arquivo escrevendo `vendedor,meta` apenas, desfaça a mudança com `git restore`, leia o arquivo para conferir que a meta de 55000 voltou e liste o histórico com `git log --oneline`.

In [28]:
# sua resposta aqui


In [29]:
# Resposta
(pasta_exercicio / "metas.csv").write_text("vendedor,meta\n", encoding="utf-8")
!git -C exercicio_casa restore metas.csv
print((pasta_exercicio / "metas.csv").read_text(encoding="utf-8"))
!git -C exercicio_casa log --oneline


vendedor,meta
Ana,55000



e72264c Atualiza meta da Ana para 55000
cb1f0e3 Adiciona metas de vendas


### Exercício 4: para pensar

Escreva em duas ou três frases: numa equipe de três analistas que montam o mesmo relatório todo mês, **o que o Git resolveria** que a pasta de rede com cópias `_v2_final` não resolve?

<details>
<summary>👀 Ver a resposta</summary>

Cada alteração fica registrada com autor, data e motivo, então dá para saber quem mudou o quê. Dá para voltar a qualquer versão anterior, e três pessoas conseguem trabalhar ao mesmo tempo, juntando o trabalho depois, sem sobrescrever o do outro.

</details>

## 🚀 Próxima parte

Foi só a porta de entrada. Para praticar, deixei no **notebook complementar** um passo a passo de branch, merge e `.gitignore`.

☕ **Pausa.** Na volta, começamos o **pandas**: tabelas com milhares de linhas e colunas de tipos diferentes, que é onde o dia a dia do analista acontece.